In [ ]:
"""
Waste Type Identification - Generazione split train/val
----------------------------------------------------------------------
"""

import re
import csv
import random
from pathlib import Path
from collections import defaultdict, Counter

# Monta il Drive
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except Exception as e:
    print("Mount Drive non eseguito:", e)

DATASET_DIR = Path('/content/drive/MyDrive/2026_MLinf_gr41/Waste-Project/dataset')
SPLITS_DIR  = Path('/content/drive/MyDrive/2026_MLinf_gr41/Waste-Project/splits')
VAL_RATIO   = 0.20                                                              # Split 80 (train) - 20 (validation)
SEED        = 1234                                                              # Per la riproducibilità

# Mappatura classe-label
MACRO_TO_LABEL = {
    'battery': 0, 'clothing': 1, 'glass': 2, 'metal': 3,
    'organic': 4, 'papery': 5, 'plastic': 6, 'undifferentiated': 7,
}
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

random.seed(SEED)
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

def sub_label_from_name(stem: str) -> str:
    """Prende le lettere iniziali del nome file: 'green1' -> 'green'. In questo modo, estrae la classe a partire dal nome del file"""
    m = re.match(r'^([a-zA-Z]+)', stem)
    return m.group(1).lower() if m else stem.lower()

records = []  # (relpath, macro, label, sub)
for macro, label in MACRO_TO_LABEL.items():
    folder = DATASET_DIR / macro
    if not folder.exists():
        print(f"ATTENZIONE: cartella mancante: {folder}")
        continue
    for p in folder.rglob('*'):
        if p.suffix.lower() in IMG_EXTS:
            sub = sub_label_from_name(p.stem)
            rel = p.relative_to(DATASET_DIR).as_posix()
            records.append((rel, macro, label, sub))

print(f"Totale immagini trovate: {len(records)}")                               # Numero di immagini totali del DataSet
# Stampa tutte le sottoclassi
print("Sotto-classi rilevate:")
for (macro, sub), n in sorted(Counter((r[1], r[3]) for r in records).items()):
    print(f"   {macro:16s} / {sub:14s}  {n:5d}")
print()

# Split stratificato
# NOTA. È fondamentale mantenere le stesse proporzioni del DataSet originale, in modo da non influenzare la balanced accuracy, in modo da ottenere risultati attendibili
groups = defaultdict(list)
for r in records:
    groups[(r[1], r[3])].append(r)

train_set, val_set = [], []
for key, items in groups.items():
    random.shuffle(items)
    # Almeno 1 in val se il gruppo ha >= 2 campioni
    n_val = max(1, round(len(items) * VAL_RATIO)) if len(items) >= 2 else 0
    val_set.extend(items[:n_val])
    train_set.extend(items[n_val:])

# Scrittura del CSV
out_csv = SPLITS_DIR / 'split.csv'
with open(out_csv, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['filepath', 'label', 'macro_label', 'sub_label', 'split'])      # Come richiesto dalla traccia, non si riportano i dati bensì esclusivamente le etichette e lo split
    for rel, macro, label, sub in train_set:
        w.writerow([rel, label, macro, sub, 'train'])
    for rel, macro, label, sub in val_set:
        w.writerow([rel, label, macro, sub, 'val'])
print(f"Salvato: {out_csv}   (train={len(train_set)}, val={len(val_set)})\n")

# Riepilogo
print("=== Conteggi per macro-classe ===")
tr_macro = Counter(r[1] for r in train_set)
vl_macro = Counter(r[1] for r in val_set)
print(f"  {'classe':16s} {'train':>7s} {'val':>6s} {'val%':>6s}")
for macro in MACRO_TO_LABEL:
    t, v = tr_macro[macro], vl_macro[macro]
    pct = 100 * v / (t + v) if (t + v) else 0
    print(f"  {macro:16s} {t:7d} {v:6d} {pct:5.1f}%")

Mounted at /content/drive
Totale immagini trovate: 15515
Sotto-classi rilevate:
   battery          / battery           945
   clothing         / clothes          5325
   clothing         / shoes            1977
   glass            / brown             607
   glass            / green             629
   glass            / transparent       775
   metal            / metal             769
   organic          / organic           985
   papery           / cardboard         891
   papery           / paper            1050
   plastic          / plastic           865
   undifferentiated / undifferentiated    697

Salvato: /content/drive/MyDrive/2026_MLinf_gr41/Waste-Project/splits/split.csv   (train=12413, val=3102)

=== Conteggi per macro-classe ===
  classe             train    val   val%
  battery              756    189  20.0%
  clothing            5842   1460  20.0%
  glass               1609    402  20.0%
  metal                615    154  20.0%
  organic              788    197  20.0%
  p